# AgriEdge-VLM — full pipeline, one notebook

Teacher distillation (AgriChat 7B) -> small offline mobile model (SigLIP classifier + SmolLM2-360M SLM).
Every cell here is load-bearing — no diagnostics, no dead-end attempts. Run top to bottom, fresh Kaggle
session, no leftover state assumed.

**Fixes folded in from prior debugging:**
- Teacher loads **4-bit only** — FP16 (16GB) killed the kernel outright on 2xT4, OOM wasn't even catchable.
- AgML images are **symlinked**, not copied — copying doubled disk and filled it after ~27 datasets.
- Image path resolution uses a **recursive filename index + case-insensitive lookup** — jsonl paths
  don't match AgML's actual folder depth (`classification/`/`detection/` segments don't exist on disk)
  and file extensions differ in case (`.jpg` vs `.JPG`) for some sources.
- Classifier labels come from the **dataset/folder structure itself** (crop = dataset name, disease =
  class subfolder), not from regex-matching the teacher's free-text answer — the text-heuristic version
  only kept ~48% of records and the folder labels are the actual ground truth anyway.


In [4]:
# =====================================================================
# CONFIGURATION — single source of truth.
# =====================================================================
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# ---- HF source for AgriMM annotations ----
HF_ANNOTATION_REPO = "boudiafA/AgriChat"
HF_ANNOTATION_SUBDIR = "dataset"
HF_ANNOTATION_REPO_TYPE = "model"
HF_TRAIN_FILENAME_CANDIDATES = ["train.jsonl", "stage2_train.jsonl"]
HF_TEST_FILENAME_CANDIDATES  = ["test.jsonl", "stage2_test.jsonl"]

# ---- Teacher ----
BASE_MODEL = "llava-hf/llava-onevision-qwen2-7b-ov-hf"
AGRICHAT_ADAPTER = "boudiafA/AgriChat"

# ---- Paths ----
PROJECT_ROOT = "/kaggle/working/agrichat_distillation"
TRAIN_JSONL = f"{PROJECT_ROOT}/data/train.jsonl"
TEST_JSONL  = f"{PROJECT_ROOT}/data/test.jsonl"
IMAGE_ROOT = f"{PROJECT_ROOT}/datasets_sorted"
TEACHER_OUTPUT = f"{PROJECT_ROOT}/teacher_outputs"
DISTILL_OUT = f"{TEACHER_OUTPUT}/distillation_train.jsonl"
STUDENT_TRAIN_JSONL = f"{TEACHER_OUTPUT}/student_train_instructions.jsonl"

EDGE_ROOT = "/kaggle/working/agriedge_vlm"
STRUCTURED_JSONL = f"{EDGE_ROOT}/data/structured_train.jsonl"
CLASSIFIER_CKPT_DIR = f"{EDGE_ROOT}/vision_classifier"
SLM_CKPT_DIR = f"{EDGE_ROOT}/slm"
EXPORT_DIR = f"{EDGE_ROOT}/export"

for d in [f"{PROJECT_ROOT}/data", TEACHER_OUTPUT, f"{EDGE_ROOT}/data",
          CLASSIFIER_CKPT_DIR, SLM_CKPT_DIR, EXPORT_DIR]:
    os.makedirs(d, exist_ok=True)

# ---- Teacher pilot ----
PILOT_SAMPLES = 1000

# ---- Vision classifier ----
VISION_MODEL_ID = "google/siglip-base-patch16-224"
CLASSIFIER_BATCH_SIZE = 32
CLASSIFIER_EPOCHS = 3
CLASSIFIER_LR = 3e-4

# ---- SLM ----
LM_MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
SLM_BATCH_SIZE = 4
SLM_GRAD_ACCUM = 4
SLM_EPOCHS = 2
SLM_LR = 2e-5
SLM_MAX_LENGTH = 512

print("Config loaded. PROJECT_ROOT:", PROJECT_ROOT, "| EDGE_ROOT:", EDGE_ROOT)


Config loaded. PROJECT_ROOT: /kaggle/working/agrichat_distillation | EDGE_ROOT: /kaggle/working/agriedge_vlm


In [5]:
# =====================================================================
# Hardware detect + install (check first, don't blind-install).
# =====================================================================
import subprocess, sys, importlib

def _ver(pkg):
    try:
        return getattr(importlib.import_module(pkg), "__version__", "unknown")
    except ImportError:
        return None

torch_ver = _ver("torch")
if torch_ver:
    import torch
    print("torch:", torch.__version__, "| CUDA avail:", torch.cuda.is_available(),
          "| GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        g = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {g.name} — {g.total_memory/1024**3:.1f} GB")
    print("NOTE: NOT a unified pool — each GPU capped at its own VRAM.")

REQUIRED = {
    "transformers": "4.45.0", "accelerate": "0.34.0", "peft": "0.13.0",
    "bitsandbytes": "0.43.0", "datasets": "2.20.0", "sentencepiece": "0.1.99",
    "safetensors": "0.4.0", "Pillow": "10.0.0", "timm": "1.0.0",
    "onnx": "1.16.0", "onnxruntime": "1.18.0",
}
IMPORT_NAME_OVERRIDES = {"Pillow": "PIL"}

def _ge(v1, v2):
    def parts(v): return [int(x) for x in v.split(".")[:3] if x.isdigit()]
    return parts(v1) >= parts(v2)

to_install = []
for pkg, min_ver in REQUIRED.items():
    v = _ver(IMPORT_NAME_OVERRIDES.get(pkg, pkg))
    if v is None or not _ge(v, min_ver):
        to_install.append(pkg)
        print(f"{pkg}: {'not installed' if v is None else f'{v} < {min_ver}'} -> install")
    else:
        print(f"{pkg}: {v} OK")

if to_install:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *to_install], check=True)
print("\nDone. If torch itself just changed, restart the kernel before continuing.")


torch: 2.10.0+cu128 | CUDA avail: True | GPU count: 2
  GPU 0: Tesla T4 — 14.6 GB
  GPU 1: Tesla T4 — 14.6 GB
NOTE: NOT a unified pool — each GPU capped at its own VRAM.
transformers: 5.0.0 OK
accelerate: 1.13.0 OK
peft: 0.19.1 OK
bitsandbytes: not installed -> install
datasets: 5.0.0 OK
sentencepiece: 0.2.1 OK
safetensors: 0.7.0 OK
Pillow: 11.3.0 OK
timm: 1.0.26 OK
onnx: 1.22.0 OK
onnxruntime: 1.30.0 OK
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.9 MB/s eta 0:00:00

Done. If torch itself just changed, restart the kernel before continuing.


In [6]:
# =====================================================================
# Download AgriMM train/test annotation JSONLs from HF.
# =====================================================================
from huggingface_hub import HfApi, hf_hub_download
import shutil

api = HfApi()
dataset_files = [f for f in api.list_repo_files(repo_id=HF_ANNOTATION_REPO, repo_type=HF_ANNOTATION_REPO_TYPE)
                  if f.startswith(f"{HF_ANNOTATION_SUBDIR}/")]
print(f"Found {len(dataset_files)} files under {HF_ANNOTATION_REPO}/{HF_ANNOTATION_SUBDIR}/")

def _resolve_and_download(candidates, local_target):
    match = next((f"{HF_ANNOTATION_SUBDIR}/{c}" for c in candidates
                  if f"{HF_ANNOTATION_SUBDIR}/{c}" in dataset_files), None)
    if match is None:
        raise FileNotFoundError(f"None of {candidates} found. Actual files: {dataset_files}")
    local_path = hf_hub_download(repo_id=HF_ANNOTATION_REPO, repo_type=HF_ANNOTATION_REPO_TYPE, filename=match)
    shutil.copy(local_path, local_target)
    print(f"{match} -> {local_target}")

_resolve_and_download(HF_TRAIN_FILENAME_CANDIDATES, TRAIN_JSONL)
_resolve_and_download(HF_TEST_FILENAME_CANDIDATES, TEST_JSONL)


Found 3 files under boudiafA/AgriChat/dataset/


dataset/train.jsonl:   0%|          | 0.00/200M [00:00<?, ?B/s]

dataset/train.jsonl -> /kaggle/working/agrichat_distillation/data/train.jsonl


dataset/test.jsonl:   0%|          | 0.00/12.4M [00:00<?, ?B/s]

dataset/test.jsonl -> /kaggle/working/agrichat_distillation/data/test.jsonl


In [4]:
# =====================================================================
# AgML-covered sources -> datasets_sorted/, SYMLINKED (not copied — copying
# doubled disk and filled it after ~27 datasets last time).
# =====================================================================
import os

try:
    import agml
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "agml"], check=True)
    import agml

os.makedirs(IMAGE_ROOT, exist_ok=True)

AGML_ALL_NAMES = [
    "almond_bloom_2023", "almond_harvest_2021", "apple_detection_drone_brazil",
    "apple_detection_spain", "apple_detection_usa", "embrapa_wgisd_grape_detection",
    "fruit_detection_worldwide", "grape_detection_californiaday", "grape_detection_californianight",
    "grape_detection_syntheticday", "mango_detection_australia", "strawberry_detection_2022",
    "strawberry_detection_2023", "tomato_ripeness_detection",
    "ghai_broccoli_detection", "ghai_green_cabbage_detection", "ghai_iceberg_lettuce_detection",
    "ghai_romaine_detection", "wheat_head_counting",
    "gemini_flower_detection_2022", "gemini_leaf_detection_2022", "gemini_plant_detection_2022",
    "gemini_pod_detection_2022", "plant_doc_detection",
    "arabica_coffee_leaf_disease_classification", "banana_leaf_disease_classification",
    "bean_disease_uganda", "betel_leaf_disease_classification",
    "blackgram_plant_leaf_disease_classification", "chilli_leaf_classification",
    "coconut_tree_disease_classification", "corn_maize_leaf_disease", "crop_weeds_greece",
    "cucumber_disease_classification", "guava_disease_pakistan",
    "java_plum_leaf_disease_classification", "leaf_counting_denmark", "onion_leaf_classification",
    "orange_leaf_disease_classification", "paddy_disease_classification",
    "papaya_leaf_disease_classification", "plant_doc_classification", "plant_seedlings_aarhus",
    "plant_village_classification", "rangeland_weeds_australia", "rice_leaf_disease_classification",
    "riseholme_strawberry_classification_2021", "soybean_weed_uav_brazil", "sugarcane_damage_usa",
    "sunflower_disease_classification", "tea_leaf_disease_classification", "tomato_leaf_disease",
    "vine_virus_photo_dataset",
]

def agml_download(name):
    dest = os.path.join(IMAGE_ROOT, name)
    if os.path.exists(dest):
        return True
    try:
        agml.data.AgMLDataLoader(name)
        src_dir = os.path.expanduser(f"~/.agml/datasets/{name}")
        if not os.path.isdir(src_dir):
            return False
        os.symlink(src_dir, dest)
        return True
    except Exception as e:
        print(f"[fail] {name}: {e}")
        return False

results = {n: agml_download(n) for n in AGML_ALL_NAMES}
ok = sum(results.values())
print(f"AgML download: {ok}/{len(AGML_ALL_NAMES)} succeeded.")
if ok < len(AGML_ALL_NAMES):
    print("Failed:", [n for n, v in results.items() if not v])
subprocess.run(["df", "-h", "/kaggle/working"])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 778.7/778.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: rich
    Found existing installation: rich 13.9.4
    Uninstalling rich-13.9.4:
      Successfully uninstalled rich-13.9.4


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.39.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.
pyiceberg 0.11.1 requires rich<15.0.0,>=10.11.0, but you have rich 15.0.0 which is incompatible.


Output()

[AgML Download]: Downloading dataset `almond_bloom_2023` to /root/.agml/datasets/almond_bloom_2023.

[AgML Download]: Extracting files for almond_bloom_2023...

...

╭─────────────────────────────── Copyright, Citation, and Documenation Information ───────────────────────────────╮
│ Dataset: vine_virus_photo_dataset                                                                               │
│                                                                                                                 │
│ You have just downloaded vine_virus_photo_dataset                                                               │
│                                                                                                                 │
│ This dataset is licensed under Apache 2.0  To learn more about this license, visit                              │
│ https://www.apache.org/licenses/LICENSE-2.0                                                                     │
│                                                                                                                 │
│ This dataset has no associated citation.                                                                        │
│                                                                                                                 │
│ You can find additional information about this dataset at:                                                      │
╰─────────────────────────────────────── Dataset: vine_virus_photo_dataset ───────────────────────────────────────╯

╭───────────────────────────────────────────────────── Note ──────────────────────────────────────────────────────╮
│                                                                                                                 │
│ This message will not be automatically shown again. To view this message again,  in an AgMLDataLoader run       │
│ `loader.info.citation_summary()`  Otherwise, you can use `agml.data.source(<dataset_name>).citation_summary()`. │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 You can find your dataset at /root/.agml/datasets/vine_virus_photo_dataset.

AgML download: 53/53 succeeded.
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  203M   20G   2% /kaggle/working


CompletedProcess(args=['df', '-h', '/kaggle/working'], returncode=0)

In [9]:
import os
from pathlib import Path

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

print("Indexing images under", IMAGE_ROOT, "...")
_image_index = {}
_n_indexed = 0
for root, dirs, files in os.walk(IMAGE_ROOT, followlinks=True):
    for fname in files:
        if os.path.splitext(fname)[1].lower() in IMAGE_EXTENSIONS:
            p = Path(root) / fname
            _image_index.setdefault(fname.lower(), []).append(p)
            _n_indexed += 1
print(f"Indexed {_n_indexed} images, {len(_image_index)} unique lowercase filenames.")

def resolve_image_path(image_path, image_root=IMAGE_ROOT):
    rel = str(image_path).replace("\\", "/")
    while rel.startswith("./"):
        rel = rel[2:]
    parts = rel.split("/")
    if parts and parts[0] == "datasets_sorted":
        parts = parts[1:]
    if parts and parts[0] in ("classification", "detection"):
        parts = parts[1:]

    candidate = Path(image_root).joinpath(*parts)
    if candidate.exists():
        return candidate

    fname = Path(rel).name.lower()
    matches = _image_index.get(fname, [])
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        scored = []
        for m in matches:
            m_str = str(m).replace("\\", "/").lower()
            score = sum(1 for part in parts if part.lower() in m_str)
            scored.append((score, m))
        scored.sort(key=lambda x: x[0], reverse=True)
        return scored[0][1]

    return candidate

print("resolve_image_path ready.")

Indexing images under /kaggle/working/agrichat_distillation/datasets_sorted ...
Indexed 276414 images, 190547 unique lowercase filenames.
resolve_image_path ready.


In [10]:
# =====================================================================
# Validate a sample against the resolver before spending compute.
# =====================================================================
from datasets import load_dataset

CHECK_LIMIT = 10000
missing, checked, missing_inatag = 0, 0, 0
train_stream = load_dataset("json", data_files={"train": TRAIN_JSONL}, split="train", streaming=True)
for record in train_stream:
    for image in record.get("images", []):
        checked += 1
        if not resolve_image_path(image).exists():
            missing += 1
            if "iNatAg_subset" in image:
                missing_inatag += 1
        if checked >= CHECK_LIMIT:
            break
    if checked >= CHECK_LIMIT:
        break

print(f"Checked: {checked}  Missing: {missing} ({100*missing/max(checked,1):.1f}%)")
print(f"  of which iNatAg_subset (never downloaded, expected): {missing_inatag}")
print(f"  other missing (manual-only sources like DRPD/GWHD2021, expected small): {missing - missing_inatag}")


Checked: 10000  Missing: 4574 (45.7%)
  of which iNatAg_subset (never downloaded, expected): 3932
  other missing (manual-only sources like DRPD/GWHD2021, expected small): 642


In [11]:
# =====================================================================
# Load teacher — 4-bit NF4 ONLY. FP16 (16GB) hard-crashed the kernel
# (OOM kill, not a catchable exception) — do not attempt FP16 here.
# =====================================================================
import torch
from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
from PIL import Image

processor = AutoProcessor.from_pretrained(BASE_MODEL)

teacher_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
base_model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    BASE_MODEL, quantization_config=teacher_bnb_config, low_cpu_mem_usage=True, device_map="auto",
)
teacher = PeftModel.from_pretrained(base_model, AGRICHAT_ADAPTER)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

print("Teacher loaded (4-bit NF4).")


processor_config.json:   0%|          | 0.00/178 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

The image processor of type `LlavaOnevisionImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['model.image_newline']
  warnings.warn(


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/1.36G [00:00<?, ?B/s]

Teacher loaded (4-bit NF4).


In [12]:
# =====================================================================
# Teacher pilot generation — resumable JSONL.
# =====================================================================
import json

def load_already_processed(path):
    done = set()
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    done.add(json.loads(line)["image"])
                except Exception:
                    continue
    return done

def run_teacher_generation(max_samples):
    already = load_already_processed(DISTILL_OUT)
    print(f"Resuming: {len(already)} already done.")
    stream = load_dataset("json", data_files={"train": TRAIN_JSONL}, split="train", streaming=True)
    produced = len(already)
    with open(DISTILL_OUT, "a", encoding="utf-8") as out_f:
        for record in stream:
            if produced >= max_samples:
                break
            imgs = record.get("images", [])
            if not imgs or imgs[0] in already:
                continue
            img_path = resolve_image_path(imgs[0])
            if not img_path.exists():
                continue

            question, ground_truth = None, None
            for msg in record.get("messages", []):
                if msg.get("role") == "user" and question is None:
                    c = msg.get("content")
                    question = c if isinstance(c, str) else next(
                        (b.get("text") for b in c if isinstance(b, dict) and b.get("type") == "text"), None)
                if msg.get("role") == "assistant" and ground_truth is None:
                    ground_truth = msg.get("content")
            if question is None or ground_truth is None:
                continue

            try:
                image = Image.open(img_path).convert("RGB")
                conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
                text = processor.apply_chat_template(conversation, add_generation_prompt=True)
                inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True)
                inputs = {k: (v.to(teacher.device) if isinstance(v, torch.Tensor) else v) for k, v in inputs.items()}
                with torch.inference_mode():
                    output_ids = teacher.generate(**inputs, max_new_tokens=256, do_sample=False)
                input_len = inputs["input_ids"].shape[1]
                teacher_answer = processor.tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                continue

            out_f.write(json.dumps({"image": imgs[0], "question": question,
                                     "ground_truth": ground_truth, "teacher_answer": teacher_answer},
                                    ensure_ascii=False) + "\n")
            out_f.flush()
            produced += 1
            if produced % 50 == 0:
                print(f"{produced}/{max_samples}")
    print(f"Done. Total: {produced}")
    return produced

run_teacher_generation(max_samples=PILOT_SAMPLES)


Resuming: 0 already done.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-e

50/1000


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for

100/1000


...

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for

950/1000


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for

1000/1000
Done. Total: 1000


1000

In [13]:
# =====================================================================
# Build student instruction dataset (ground truth + teacher answer).
# =====================================================================
with open(DISTILL_OUT, "r", encoding="utf-8") as f_in, \
     open(STUDENT_TRAIN_JSONL, "w", encoding="utf-8") as f_out:
    for line in f_in:
        r = json.loads(line)
        f_out.write(json.dumps({
            "image": r["image"],
            "messages": [
                {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": r["question"]}]},
                {"role": "assistant", "content": r["ground_truth"]},
            ],
            "teacher_answer_reference": r["teacher_answer"],
        }, ensure_ascii=False) + "\n")

print("Student training file:", STUDENT_TRAIN_JSONL)


Student training file: /kaggle/working/agrichat_distillation/teacher_outputs/student_train_instructions.jsonl


## AgriEdge — small offline model

Classifier labels come from the **dataset/folder structure** (real ground truth), not from parsing
the teacher's free text. Crop = dataset name; disease = class subfolder, only for datasets that are
actually disease-labeled (avoids manufacturing fake "healthy" labels for species/counting datasets).


In [14]:
# =====================================================================
# Build structured records: folder-derived crop/disease (ground truth) +
# light text-extracted severity/evidence (descriptive only, non-gating).
# =====================================================================
import re
from collections import Counter

DISEASE_DATASET_KEYWORDS = ["disease", "leaf_classification", "leaf_disease", "plant_village"]

def looks_like_disease_dataset(dataset_name):
    name = dataset_name.lower()
    return any(k in name for k in DISEASE_DATASET_KEYWORDS)

def normalize_label(label):
    label = str(label).strip().lower().replace("-", "_").replace(" ", "_")
    while "__" in label:
        label = label.replace("__", "_")
    return label

SEVERITY_TERMS = {
    "mild": ["mild", "slight", "few spots", "early stage"],
    "moderate": ["moderate", "several", "spreading"],
    "severe": ["severe", "heavy", "extensive", "widespread", "advanced"],
}
EVIDENCE_TERMS = ["brown spots", "yellow halos", "yellowing", "wilting", "black spots",
                   "white powdery", "curling", "discoloration", "necrosis", "lesions",
                   "spots", "holes", "mottling"]

def extract_severity_evidence(text):
    t = text.lower()
    severity = next((s for s, terms in SEVERITY_TERMS.items() if any(k in t for k in terms)), "unspecified")
    evidence = [e for e in EVIDENCE_TERMS if e in t]
    return severity, evidence

structured_records = []
missing_img = 0
with open(STUDENT_TRAIN_JSONL, "r", encoding="utf-8") as f_in:
    for line in f_in:
        r = json.loads(line)
        img_path = resolve_image_path(r["image"])
        if not img_path.exists():
            missing_img += 1
            continue

        rel_to_root = img_path.relative_to(Path(IMAGE_ROOT))
        parts = rel_to_root.parts
        dataset_name = parts[0] if len(parts) >= 1 else "unknown"
        folder_label = parts[-2] if len(parts) >= 3 else None

        crop = normalize_label(dataset_name)
        if looks_like_disease_dataset(dataset_name) and folder_label:
            disease = normalize_label(folder_label)
        else:
            disease = "not_applicable"

        gt = r["messages"][1]["content"]
        question = r["messages"][0]["content"][1]["text"]
        teacher_answer = r.get("teacher_answer_reference", "")
        severity, evidence = extract_severity_evidence(f"{gt} {teacher_answer}")

        structured_records.append({
            "image": r["image"], "resolved_image_path": str(img_path),
            "question": question, "ground_truth": gt, "teacher_answer": teacher_answer,
            "crop": crop, "disease": disease, "visual_evidence": evidence,
            "severity": severity, "recommended_action": gt,
        })

with open(STRUCTURED_JSONL, "w", encoding="utf-8") as f_out:
    for rec in structured_records:
        f_out.write(json.dumps(rec, ensure_ascii=False) + "\n")

crop_counts = Counter(r["crop"] for r in structured_records)
disease_counts = Counter(r["disease"] for r in structured_records)
print(f"Structured records: {len(structured_records)}  (missing images skipped: {missing_img})")
print(f"Crop classes: {len(crop_counts)}  Disease classes: {len(disease_counts)}")
print("\nCrop distribution:")
for c, n in crop_counts.most_common():
    print(f"  {c:<45} {n:>5}")
print("\nDisease distribution:")
for d, n in disease_counts.most_common():
    print(f"  {d:<45} {n:>5}")


Structured records: 1000  (missing images skipped: 0)
Crop classes: 32  Disease classes: 93

Crop distribution:
  arabica_coffee_leaf_disease_classification      207
  plant_village_classification                    187
  wheat_head_counting                             129
  soybean_weed_uav_brazil                          68
  rangeland_weeds_australia                        61
  paddy_disease_classification                     34
  chilli_leaf_classification                       32
  apple_detection_usa                              23
  leaf_counting_denmark                            21
  tea_leaf_disease_classification                  20
  coconut_tree_disease_classification              17
  mango_detection_australia                        15
  apple_detection_spain                            15
  riseholme_strawberry_classification_2021         14
  onion_leaf_classification                        14
  corn_maize_leaf_disease                          14
  rice_leaf_disease_clas

In [19]:
# =====================================================================
# Vision classifier: SigLIP backbone (frozen except last 2 layers) +
# crop head + disease head.
# FIX: backbone kept in fp32 (mixed precision is handled by autocast in training)
# =====================================================================
import os, random
import torch
import torch.nn as nn
from collections import Counter
from PIL import Image
from transformers import SiglipVisionModel, AutoImageProcessor
from torch.utils.data import Dataset

vision_processor = AutoImageProcessor.from_pretrained(VISION_MODEL_ID)
vision_backbone = SiglipVisionModel.from_pretrained(VISION_MODEL_ID).float().cuda()  # fp32 master weights

for p in vision_backbone.parameters():
    p.requires_grad_(False)
for layer in vision_backbone.vision_model.encoder.layers[-2:]:
    for p in layer.parameters():
        p.requires_grad_(True)

class AgriClassifier(nn.Module):
    def __init__(self, backbone, hidden_dim, n_crop, n_disease):
        super().__init__()
        self.backbone = backbone
        self.crop_head = nn.Linear(hidden_dim, n_crop)
        self.disease_head = nn.Linear(hidden_dim, n_disease)

    def forward(self, pixel_values):
        pooled = self.backbone(pixel_values=pixel_values).pooler_output.float()
        return self.crop_head(pooled), self.disease_head(pooled)

crop_labels = sorted({r["crop"] for r in structured_records})
disease_labels = sorted({r["disease"] for r in structured_records})
crop_to_idx = {c: i for i, c in enumerate(crop_labels)}
disease_to_idx = {d: i for i, d in enumerate(disease_labels)}

hidden_dim = vision_backbone.config.hidden_size
classifier = AgriClassifier(vision_backbone, hidden_dim, len(crop_labels), len(disease_labels)).cuda()
n_params = sum(p.numel() for p in classifier.parameters())
n_trainable = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
print(f"Classifier: {n_params/1e6:.1f}M total, {n_trainable/1e6:.1f}M trainable, "
      f"{len(crop_labels)} crop classes, {len(disease_labels)} disease classes")

class ClassifierDataset(Dataset):
    def __init__(self, records, processor):
        self.records = records
        self.processor = processor

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        image = Image.open(r["resolved_image_path"]).convert("RGB")
        pixel_values = self.processor(images=image, return_tensors="pt")["pixel_values"][0]
        return {"pixel_values": pixel_values, "crop_label": crop_to_idx[r["crop"]],
                "disease_label": disease_to_idx[r["disease"]]}

random.seed(42)
shuffled = structured_records[:]
random.shuffle(shuffled)
split_i = int(0.9 * len(shuffled))
clf_train_records, clf_val_records = shuffled[:split_i], shuffled[split_i:]
print(f"Train: {len(clf_train_records)}  Val: {len(clf_val_records)}")

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.final_layer_norm.bias                     

Classifier: 93.0M total, 14.3M trainable, 32 crop classes, 93 disease classes
Train: 900  Val: 100


In [20]:
# =====================================================================
# Train classifier — sqrt-inverse-frequency weighted disease loss,
# mixed precision (fp32 weights + autocast), grad clipping, NaN guards,
# resumable, OOM-safe.
# =====================================================================
from torch.utils.data import DataLoader

train_ds = ClassifierDataset(clf_train_records, vision_processor)
val_ds = ClassifierDataset(clf_val_records, vision_processor)
train_loader = DataLoader(train_ds, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=False, num_workers=2)

# Sanity check: inputs must be finite before blaming the model
_b = next(iter(train_loader))
assert torch.isfinite(_b["pixel_values"]).all(), "NaN/inf in pixel_values - check image loading"
del _b

# --- class weights for the disease loss ---
train_disease_counts = Counter(r["disease"] for r in clf_train_records)
disease_weights = torch.tensor(
    [1.0 / (train_disease_counts.get(d, 0) ** 0.5) if train_disease_counts.get(d, 0) > 0 else 0.0
     for d in disease_labels], dtype=torch.float32)
nonzero = disease_weights > 0
if nonzero.any():
    disease_weights[nonzero] /= disease_weights[nonzero].mean()
disease_weights = disease_weights.cuda()

crop_loss_fn = nn.CrossEntropyLoss()
disease_loss_fn = nn.CrossEntropyLoss(weight=disease_weights)

# --- mixed precision: bf16 if supported (no scaler needed), else fp16 + GradScaler ---
use_bf16 = torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=not use_bf16)
print(f"Autocast dtype: {amp_dtype}, GradScaler: {'off' if use_bf16 else 'on'}")

# --- optimizer: gentler LR for the unfrozen backbone layers, higher for the fresh heads ---
BACKBONE_LR = CLASSIFIER_LR
HEAD_LR = CLASSIFIER_LR * 10
backbone_params = [p for p in classifier.backbone.parameters() if p.requires_grad]
head_params = list(classifier.crop_head.parameters()) + list(classifier.disease_head.parameters())
trainable_params = backbone_params + head_params
optimizer = torch.optim.AdamW(
    [{"params": backbone_params, "lr": BACKBONE_LR},
     {"params": head_params, "lr": HEAD_LR}],
    weight_decay=0.01)

# --- resume (ignores stale/NaN checkpoints from the old fp16 run) ---
CKPT_FORMAT = "fp32_amp_v2"
resume_path = f"{CLASSIFIER_CKPT_DIR}/latest.pt"
start_epoch = 0
if os.path.exists(resume_path):
    ckpt = torch.load(resume_path, map_location="cuda")
    finite = all(torch.isfinite(v).all() for v in ckpt["model_state"].values() if v.is_floating_point())
    if ckpt.get("format") == CKPT_FORMAT and finite:
        classifier.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optim_state"])
        start_epoch = ckpt["epoch"] + 1
        print(f"Resumed from epoch {ckpt['epoch']}")
    else:
        print("Ignoring old/invalid checkpoint - starting fresh")
    del ckpt

for epoch in range(start_epoch, CLASSIFIER_EPOCHS):
    classifier.train()
    total_loss, steps, skipped, ooms = 0.0, 0, 0, 0
    for batch in train_loader:
        try:
            pixel_values = batch["pixel_values"].cuda()          # fp32; autocast handles casting
            crop_b, disease_b = batch["crop_label"].cuda(), batch["disease_label"].cuda()

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=amp_dtype):
                crop_logits, disease_logits = classifier(pixel_values)
            loss = crop_loss_fn(crop_logits.float(), crop_b) + disease_loss_fn(disease_logits.float(), disease_b)

            if not torch.isfinite(loss):
                skipped += 1
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            steps += 1
        except torch.cuda.OutOfMemoryError:
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            ooms += 1
            continue

    classifier.eval()
    correct_crop, correct_disease, total = 0, 0, 0
    with torch.no_grad(), torch.autocast("cuda", dtype=amp_dtype):
        for batch in val_loader:
            pixel_values = batch["pixel_values"].cuda()
            crop_logits, disease_logits = classifier(pixel_values)
            correct_crop += (crop_logits.argmax(-1).cpu() == batch["crop_label"]).sum().item()
            correct_disease += (disease_logits.argmax(-1).cpu() == batch["disease_label"]).sum().item()
            total += batch["crop_label"].size(0)

    print(f"Epoch {epoch}: loss={total_loss/max(steps,1):.4f} "
          f"val_crop_acc={correct_crop/max(total,1):.3f} val_disease_acc={correct_disease/max(total,1):.3f} "
          f"(skipped_nonfinite={skipped}, oom={ooms})")
    torch.save({"model_state": classifier.state_dict(), "optim_state": optimizer.state_dict(),
                "epoch": epoch, "format": CKPT_FORMAT}, resume_path)

print("Classifier training done.")

Autocast dtype: torch.bfloat16, GradScaler: off
Ignoring old/invalid checkpoint - starting fresh
Epoch 0: loss=4.1586 val_crop_acc=0.930 val_disease_acc=0.710 (skipped_nonfinite=0, oom=0)
Epoch 1: loss=1.6822 val_crop_acc=0.970 val_disease_acc=0.700 (skipped_nonfinite=0, oom=0)
Epoch 2: loss=0.8371 val_crop_acc=0.980 val_disease_acc=0.720 (skipped_nonfinite=0, oom=0)
Classifier training done.


In [23]:
# =====================================================================
# SLM text pairs: structured findings -> recommendation text.
# Fields can be str, list of str, or list of content blocks like
# {"type": "text", "text": "..."} -> flatten everything to plain text.
# =====================================================================
def flatten_text(x):
    """str / content-block dict / list of either / None -> list of clean strings."""
    if x is None:
        return []
    if isinstance(x, str):
        s = x.strip()
        return [s] if s else []
    if isinstance(x, dict):
        for key in ("text", "content"):
            if key in x:
                return flatten_text(x[key])
        return []                      # non-text blocks (images etc.) are ignored
    if isinstance(x, (list, tuple)):
        out = []
        for item in x:
            out.extend(flatten_text(item))
        return out
    s = str(x).strip()
    return [s] if s else []

def as_text(x):
    parts = [p if p[-1] in ".!?" else p + "." for p in flatten_text(x)]
    return " ".join(parts)

def render_prompt(rec):
    disease = rec.get("disease")
    disease_str = disease if disease and disease != "not_applicable" else "no disease classification for this crop type"
    evidence = flatten_text(rec.get("visual_evidence"))
    evidence_str = ", ".join(evidence) if evidence else "no specific markers listed"
    return (f"Crop: {rec['crop']}\nVisual finding: {evidence_str}\nPossible condition: {disease_str}\n"
            f"Severity: {rec.get('severity')}\n\nQuestion: {as_text(rec.get('question'))}\nAnswer:")

slm_pairs, dropped = [], 0
for r in structured_records:
    action = as_text(r.get("recommended_action"))
    if not action:
        dropped += 1
        continue
    slm_pairs.append((render_prompt(r), " " + action))

n_evidence = sum(1 for r in structured_records if flatten_text(r.get("visual_evidence")))
print(f"{len(slm_pairs)} SLM pairs ({dropped} dropped). Records with visual evidence: {n_evidence}/{len(structured_records)}")
print(f"Example:\n{slm_pairs[0][0]}\n---> {slm_pairs[0][1][:200]}")

1000 SLM pairs (0 dropped). Records with visual evidence: 0/1000
Example:
Crop: arabica_coffee_leaf_disease_classification
Visual finding: no specific markers listed
Possible condition: miner
Severity: unspecified

Question: Identify the plant and its specific variety if mentioned.
Answer:
--->  The plant is *Coffea*, a genus of evergreen shrubs or small trees, commonly known as coffee plants.


In [24]:
print(repr(structured_records[0]["visual_evidence"]))
print(Counter(r["severity"] for r in structured_records).most_common(5))

[]
[('unspecified', 1000)]


In [25]:
# =====================================================================
# SLM — full fine-tune. fp32 master weights + autocast (was: pure fp16 -> NaN).
# 360M params: ~1.4GB weights + 1.4GB grads + ~2.9GB Adam state, fits a 16GB GPU.
# =====================================================================
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import Dataset as _Dataset, DataLoader as _DataLoader

slm_tokenizer = AutoTokenizer.from_pretrained(LM_MODEL_ID)
if slm_tokenizer.pad_token is None:
    slm_tokenizer.pad_token = slm_tokenizer.eos_token

slm_model = AutoModelForCausalLM.from_pretrained(LM_MODEL_ID).float().cuda()
slm_model.gradient_checkpointing_enable()
slm_model.config.use_cache = False          # required with gradient checkpointing
n_slm_params = sum(p.numel() for p in slm_model.parameters())
print(f"SLM params: {n_slm_params/1e6:.1f}M")

# bf16 needs compute capability >= 8 (Ampere+). On T4/P100 use fp16 + GradScaler.
slm_bf16 = torch.cuda.get_device_capability()[0] >= 8
slm_amp_dtype = torch.bfloat16 if slm_bf16 else torch.float16
slm_scaler = torch.amp.GradScaler("cuda", enabled=not slm_bf16)
print(f"SLM autocast: {slm_amp_dtype}, GradScaler: {'off' if slm_bf16 else 'on'}")

class SLMDataset(_Dataset):
    def __init__(self, pairs, tokenizer, max_length):
        self.pairs, self.tokenizer, self.max_length = pairs, tokenizer, max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        prompt, target = self.pairs[idx]
        full = prompt + target + self.tokenizer.eos_token
        enc = self.tokenizer(full, truncation=True, max_length=self.max_length,
                              padding="max_length", return_tensors="pt")
        prompt_len = len(self.tokenizer(prompt, truncation=True, max_length=self.max_length)["input_ids"])
        labels = enc["input_ids"].clone()
        labels[0, :prompt_len] = -100
        labels[enc["attention_mask"] == 0] = -100
        return {"input_ids": enc["input_ids"][0], "attention_mask": enc["attention_mask"][0], "labels": labels[0]}

# Check that targets aren't being truncated away
lens = sorted(len(slm_tokenizer(p + t)["input_ids"]) for p, t in slm_pairs)
print(f"Token length: median={lens[len(lens)//2]}, max={lens[-1]}, "
      f"over SLM_MAX_LENGTH ({SLM_MAX_LENGTH}): {sum(l > SLM_MAX_LENGTH for l in lens)}")

slm_dataset = SLMDataset(slm_pairs, slm_tokenizer, SLM_MAX_LENGTH)
slm_loader = _DataLoader(slm_dataset, batch_size=SLM_BATCH_SIZE, shuffle=True)
slm_optimizer = torch.optim.AdamW(slm_model.parameters(), lr=SLM_LR, weight_decay=0.01)

SLM_CKPT_FORMAT = "fp32_amp_v2"
slm_resume_path = f"{SLM_CKPT_DIR}/latest.pt"
slm_start_epoch = 0
if os.path.exists(slm_resume_path):
    ckpt = torch.load(slm_resume_path, map_location="cuda")
    finite = all(torch.isfinite(v).all() for v in ckpt["model_state"].values() if v.is_floating_point())
    if ckpt.get("format") == SLM_CKPT_FORMAT and finite:
        slm_model.load_state_dict(ckpt["model_state"])
        slm_optimizer.load_state_dict(ckpt["optim_state"])
        slm_start_epoch = ckpt["epoch"] + 1
        print(f"Resumed SLM from epoch {ckpt['epoch']}")
    else:
        print("Ignoring old/invalid SLM checkpoint - starting fresh")
    del ckpt

for epoch in range(slm_start_epoch, SLM_EPOCHS):
    slm_model.train()
    total_loss, step_count, skipped, ooms = 0.0, 0, 0, 0
    slm_optimizer.zero_grad(set_to_none=True)
    for batch in slm_loader:
        try:
            batch = {k: v.cuda() for k, v in batch.items()}
            with torch.autocast("cuda", dtype=slm_amp_dtype):
                out = slm_model(**batch)
            loss = out.loss.float()
            if not torch.isfinite(loss):
                skipped += 1
                continue

            slm_scaler.scale(loss / SLM_GRAD_ACCUM).backward()
            total_loss += loss.item()
            step_count += 1                      # counts only successful micro-batches
            if step_count % SLM_GRAD_ACCUM == 0:
                slm_scaler.unscale_(slm_optimizer)
                torch.nn.utils.clip_grad_norm_(slm_model.parameters(), 1.0)
                slm_scaler.step(slm_optimizer)
                slm_scaler.update()
                slm_optimizer.zero_grad(set_to_none=True)
        except torch.cuda.OutOfMemoryError:
            slm_optimizer.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            ooms += 1
            continue

    print(f"SLM epoch {epoch}: avg_loss={total_loss/max(step_count,1):.4f} "
          f"(skipped_nonfinite={skipped}, oom={ooms})")
    torch.save({"model_state": slm_model.state_dict(), "optim_state": slm_optimizer.state_dict(),
                "epoch": epoch, "format": SLM_CKPT_FORMAT}, slm_resume_path)

print("SLM training done.")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

SLM params: 361.8M
SLM autocast: torch.float16, GradScaler: on
Token length: median=73, max=121, over SLM_MAX_LENGTH (512): 0
SLM epoch 0: avg_loss=0.8940 (skipped_nonfinite=0, oom=0)
SLM epoch 1: avg_loss=0.4991 (skipped_nonfinite=0, oom=0)
SLM training done.


In [26]:
# =====================================================================
# Full pipeline on TEST_JSONL — classifier -> render -> SLM. Never
# trained on this split (structured_records only ever came from
# STUDENT_TRAIN_JSONL, itself built only from TRAIN_JSONL).
# The prompt uses the SAME render_prompt() as training so the SLM sees
# the exact format it learned.
# =====================================================================
classifier.eval()
slm_model.eval()
slm_model.config.use_cache = True           # fast generation
test_records = [json.loads(l) for l in open(TEST_JSONL, encoding="utf-8")]

results = []
for r in test_records[:50]:
    imgs = r.get("images") or ([r["image"]] if "image" in r else [])
    if not imgs:
        continue
    img_path = resolve_image_path(imgs[0])
    if not img_path.exists():
        continue
    question = None
    for m in r.get("messages", []):
        if m.get("role") == "user":
            c = m.get("content")
            question = c if isinstance(c, str) else next(
                (b.get("text") for b in c if isinstance(b, dict) and b.get("type") == "text"), None)
            break
    if question is None:
        continue

    image = Image.open(img_path).convert("RGB")
    pixel_values = vision_processor(images=image, return_tensors="pt")["pixel_values"].cuda()   # fp32
    with torch.no_grad(), torch.autocast("cuda", dtype=amp_dtype):
        crop_logits, disease_logits = classifier(pixel_values)
    pred_crop = crop_labels[crop_logits.argmax(-1).item()]
    pred_disease = disease_labels[disease_logits.argmax(-1).item()]
    confidence = torch.softmax(disease_logits.float(), dim=-1).max().item()

    # Same renderer as training. The classifier gives no evidence/severity,
    # so use the same placeholders the training data uses for missing values.
    prompt = render_prompt({"crop": pred_crop, "disease": pred_disease, "visual_evidence": [],
                            "severity": "unspecified", "question": question})
    inputs = slm_tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad(), torch.autocast("cuda", dtype=slm_amp_dtype):
        out_ids = slm_model.generate(**inputs, max_new_tokens=150, do_sample=False,
                                     pad_token_id=slm_tokenizer.pad_token_id)
    answer = slm_tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    results.append({"image": imgs[0], "question": question, "pred_crop": pred_crop,
                     "pred_disease": pred_disease, "confidence": confidence, "answer": answer})

print(f"Ran on {len(results)} test images.")
if results:
    print(json.dumps(results[0], indent=2))

Ran on 21 test images.
{
  "image": "datasets_sorted\\detection\\wheat_head_counting\\images\\cbdcf874095008d8b9a73b70eaba48015d77ce9409307026265f68ea81370dff.png",
  "question": "What is the scientific name of the plant in the image?",
  "pred_crop": "wheat_head_counting",
  "pred_disease": "not_applicable",
  "confidence": 0.9916704893112183,
  "answer": "The plant is wheat, specifically a species of wheat, but the exact scientific name is not specified in the provided information."
}


In [27]:
# =====================================================================
# Export: classifier -> ONNX (+ labels & preprocessor), SLM -> HF format + GGUF instructions.
# =====================================================================
import json
os.makedirs(EXPORT_DIR, exist_ok=True)

# --- classifier -> ONNX (fp32; quantize afterwards) ---
classifier.eval()
dummy_input = torch.randn(1, 3, 224, 224, dtype=torch.float32).cuda()
onnx_path = f"{EXPORT_DIR}/agri_classifier.onnx"
export_kwargs = dict(input_names=["pixel_values"], output_names=["crop_logits", "disease_logits"],
                     dynamic_axes={"pixel_values": {0: "batch"}, "crop_logits": {0: "batch"},
                                   "disease_logits": {0: "batch"}},
                     opset_version=17)
with torch.no_grad():
    try:
        torch.onnx.export(classifier, dummy_input, onnx_path, dynamo=False, **export_kwargs)
    except TypeError:                          # older torch without the dynamo argument
        torch.onnx.export(classifier, dummy_input, onnx_path, **export_kwargs)
print("Exported:", onnx_path)

# Verify ONNX output matches PyTorch
try:
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    with torch.no_grad():
        ref_crop, ref_dis = classifier(dummy_input)
    onnx_crop, onnx_dis = sess.run(None, {"pixel_values": dummy_input.cpu().numpy()})
    print("ONNX vs torch max abs diff  crop:", float(abs(ref_crop.cpu().numpy() - onnx_crop).max()),
          " disease:", float(abs(ref_dis.cpu().numpy() - onnx_dis).max()))
except ImportError:
    print("pip install onnxruntime to verify the export")

# Label maps + image preprocessor are needed to actually use the ONNX model
with open(f"{EXPORT_DIR}/labels.json", "w") as f:
    json.dump({"crop_labels": crop_labels, "disease_labels": disease_labels}, f, indent=2)
vision_processor.save_pretrained(f"{EXPORT_DIR}/vision_processor")

# --- SLM -> HF format ---
slm_model.config.use_cache = True
slm_hf_export_dir = f"{EXPORT_DIR}/slm_hf"
slm_model.save_pretrained(slm_hf_export_dir)
slm_tokenizer.save_pretrained(slm_hf_export_dir)
print("Saved SLM to:", slm_hf_export_dir)

print(r'''
Convert SLM to GGUF (run in a shell with llama.cpp cloned):
  git clone https://github.com/ggerganov/llama.cpp
  cd llama.cpp && pip install -r requirements.txt
  python convert_hf_to_gguf.py ''' + slm_hf_export_dir + r''' --outtype f16 --outfile agriedge_slm_f16.gguf
  cmake -B build && cmake --build build --config Release -t llama-quantize
  ./build/bin/llama-quantize agriedge_slm_f16.gguf agriedge_slm_q4_k_m.gguf Q4_K_M

Classifier ONNX -> INT8: onnxruntime.quantization.quantize_dynamic
''')

/tmp/ipykernel_58/99150627.py:17: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(classifier, dummy_input, onnx_path, dynamo=False, **export_kwargs)
/usr/local/lib/python3.12/dist-packages/transformers/integrations/sdpa_attention.py:77: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  is_causal = query.shape[2] > 1 and attention_mask is None and is_causal


Exported: /kaggle/working/agriedge_vlm/export/agri_classifier.onnx
ONNX vs torch max abs diff  crop: 2.652406692504883e-06  disease: 2.2649765014648438e-06


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved SLM to: /kaggle/working/agriedge_vlm/export/slm_hf

Convert SLM to GGUF (run in a shell with llama.cpp cloned):
  git clone https://github.com/ggerganov/llama.cpp
  cd llama.cpp && pip install -r requirements.txt
  python convert_hf_to_gguf.py /kaggle/working/agriedge_vlm/export/slm_hf --outtype f16 --outfile agriedge_slm_f16.gguf
  cmake -B build && cmake --build build --config Release -t llama-quantize
  ./build/bin/llama-quantize agriedge_slm_f16.gguf agriedge_slm_q4_k_m.gguf Q4_K_M

Classifier ONNX -> INT8: onnxruntime.quantization.quantize_dynamic



In [28]:
# =====================================================================
# Final report.
# =====================================================================
def _mb(path):
    return os.path.getsize(path) / 1024**2 if os.path.exists(path) else None

print("=" * 60)
print("AGRIEDGE-VLM FINAL REPORT")
print("=" * 60)
print(f"Vision backbone: {VISION_MODEL_ID} (~{sum(p.numel() for p in vision_backbone.parameters())/1e6:.1f}M)")
print(f"Classifier heads: crop({len(crop_labels)}) + disease({len(disease_labels)})")
print(f"SLM: {LM_MODEL_ID} (~{n_slm_params/1e6:.1f}M)")
print(f"Combined FP16: ~{(sum(p.numel() for p in vision_backbone.parameters()) + n_slm_params)/1e6:.1f}M")
onnx_size = _mb(f"{EXPORT_DIR}/agri_classifier.onnx")
if onnx_size:
    print(f"Classifier ONNX (fp16): {onnx_size:.1f} MB")
print("\nRemaining before an APK: INT8-quantize classifier ONNX, run GGUF quantize commands above,")
print("wire into Android via ONNX Runtime Mobile + llama.cpp JNI bindings.")


AGRIEDGE-VLM FINAL REPORT
Vision backbone: google/siglip-base-patch16-224 (~92.9M)
Classifier heads: crop(32) + disease(93)
SLM: HuggingFaceTB/SmolLM2-360M-Instruct (~361.8M)
Combined FP16: ~454.7M
Classifier ONNX (fp16): 355.0 MB

Remaining before an APK: INT8-quantize classifier ONNX, run GGUF quantize commands above,
wire into Android via ONNX Runtime Mobile + llama.cpp JNI bindings.


In [38]:
# =====================================================================
# Push to Hugging Face Hub — auto-creates repos, writes model cards,
# uploads classifier (ONNX + labels + preprocessor) and SLM (HF format).
# =====================================================================
from huggingface_hub import HfApi, create_repo

HF_TOKEN = "HF_TOKEN_REDACTED"   # <-- paste your WRITE token here
assert HF_TOKEN.startswith("hf_") and "xxxx" not in HF_TOKEN, "Paste your real Hugging Face write token"

PRIVATE = True                       # flip to False when you're ready to publish
api = HfApi(token=HF_TOKEN)
hf_user = api.whoami()["name"]       # raises if the token is invalid
CLF_REPO = f"{hf_user}/agriedge-classifier"
SLM_REPO = f"{hf_user}/agriedge-slm"
print("Logged in as:", hf_user)

# ---- model cards ----
clf_card = f"""---
base_model: {VISION_MODEL_ID}
library_name: onnx
pipeline_tag: image-classification
tags: [agriculture, plant-disease, siglip, onnx]
---
# AgriEdge classifier (ONNX)

SigLIP vision backbone (`{VISION_MODEL_ID}`, last 2 encoder layers fine-tuned) with two linear heads:
a crop head ({len(crop_labels)} classes) and a disease head ({len(disease_labels)} classes).

- Input: `pixel_values` (batch, 3, 224, 224), float32, preprocessed with the SigLIP image processor in `vision_processor/`
- Outputs: `crop_logits`, `disease_logits`
- Class names: `labels.json`
- Held-out validation (100 images): crop acc 0.98, disease acc 0.72

Usage:

    import json, numpy as np, onnxruntime as ort
    from huggingface_hub import snapshot_download
    from transformers import AutoImageProcessor
    from PIL import Image

    path = snapshot_download("{CLF_REPO}")
    labels = json.load(open(f"{{path}}/labels.json"))
    proc = AutoImageProcessor.from_pretrained(f"{{path}}/vision_processor")
    sess = ort.InferenceSession(f"{{path}}/agri_classifier.onnx")
    x = proc(images=Image.open("leaf.jpg").convert("RGB"), return_tensors="np")["pixel_values"]
    crop_logits, disease_logits = sess.run(None, {{"pixel_values": x}})

Limitations: small training set (900 images, {len(disease_labels)} disease classes); the "crop" labels come from source
dataset names; rare disease classes are poorly covered by the validation split.
"""

slm_card = f"""---
base_model: {LM_MODEL_ID}
library_name: transformers
pipeline_tag: text-generation
tags: [agriculture, plant-disease, fine-tuned]
---
# AgriEdge SLM

Full fine-tune of `{LM_MODEL_ID}` ({n_slm_params/1e6:.0f}M parameters) on {len(slm_pairs)} prompt/answer pairs
that turn structured crop findings into text answers. It is meant to sit behind the AgriEdge classifier.

Prompt format (must match exactly):

    Crop: <crop>
    Visual finding: no specific markers listed
    Possible condition: <disease>
    Severity: unspecified

    Question: <question>
    Answer:

Limitations: trained on a small dataset; answers depend only on crop, disease label and question, so classifier
mistakes propagate into the answer, and the model often says details are "not specified". Not a substitute for
expert agronomic advice.
"""

def push(repo_id, card_text, folder, allow=None, ignore=None):
    create_repo(repo_id, repo_type="model", private=PRIVATE, exist_ok=True, token=HF_TOKEN)
    api.upload_file(path_or_fileobj=card_text.encode("utf-8"), path_in_repo="README.md",
                    repo_id=repo_id, repo_type="model", commit_message="Add model card")
    api.upload_folder(folder_path=folder, repo_id=repo_id, repo_type="model",
                      allow_patterns=allow, ignore_patterns=ignore, commit_message="Upload AgriEdge export")
    print(f"Pushed: https://huggingface.co/{repo_id}  ({'private' if PRIVATE else 'public'})")

# classifier repo: only the ONNX + labels + preprocessor (not the SLM folder)
push(CLF_REPO, clf_card, EXPORT_DIR,
     allow=["agri_classifier.onnx", "labels.json", "vision_processor/*"])

# SLM repo: the HF-format folder saved by the export cell
push(SLM_REPO, slm_card, slm_hf_export_dir)

Logged in as: vedantjadhav701


NameError: name 'crop_labels' is not defined

In [9]:
# =====================================================================
# Cell 1 — config: India-only crops
# =====================================================================
import os, sys, subprocess, json, random, shutil
from pathlib import Path

HF_TOKEN = "hf token"     # token that can read your private repos
assert HF_TOKEN.startswith("hf_") and "xxxx" not in HF_TOKEN, "Paste your real Hugging Face token"
CLF_REPO = "vedantjadhav701/agriedge-classifier"

# AgML dataset name -> crop name used as the crop label.
# Left out on purpose: bean_disease_uganda (Ugandan field images), plant_village_classification and
# plant_doc_classification (multi-crop: the crop is inside the folder name, needs a parser),
# sugarcane_damage_usa (US images), all detection/weed/counting datasets (no disease labels).
INDIA_DATASETS = {
    "paddy_disease_classification": "rice",
    "rice_leaf_disease_classification": "rice",
    "corn_maize_leaf_disease": "maize",
    "blackgram_plant_leaf_disease_classification": "black_gram",
    "tomato_leaf_disease": "tomato",
    "chilli_leaf_classification": "chilli",
    "onion_leaf_classification": "onion",
    "cucumber_disease_classification": "cucumber",
    "sunflower_disease_classification": "sunflower",
    "banana_leaf_disease_classification": "banana",
    "papaya_leaf_disease_classification": "papaya",
    "guava_disease_pakistan": "guava",                 # images from Pakistan; same crop, similar region
    "orange_leaf_disease_classification": "orange",
    "java_plum_leaf_disease_classification": "jamun",
    "coconut_tree_disease_classification": "coconut",
    "tea_leaf_disease_classification": "tea",
    "arabica_coffee_leaf_disease_classification": "coffee",
    "betel_leaf_disease_classification": "betel_leaf",
}

EDGE_ROOT = "/kaggle/working/agriedge_vlm"
IMAGE_ROOT = f"{EDGE_ROOT}/india_datasets"
CLF_JSONL = f"{EDGE_ROOT}/data/india_clf_records.jsonl"
EXPORT_DIR = f"{EDGE_ROOT}/export"

for d in [IMAGE_ROOT, f"{EDGE_ROOT}/data", EXPORT_DIR]:
    os.makedirs(d, exist_ok=True)
print(f"Config loaded: {len(INDIA_DATASETS)} datasets, {len(set(INDIA_DATASETS.values()))} crops")

Config loaded: 18 datasets, 17 crops


In [10]:
# =====================================================================
# Cell 2 — download India-only AgML datasets (symlinked into IMAGE_ROOT)
# =====================================================================
try:
    import agml
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "agml"], check=True)
    import agml

def agml_download(name):
    dest = os.path.join(IMAGE_ROOT, name)
    if os.path.exists(dest):
        return True
    try:
        agml.data.AgMLDataLoader(name)
        src_dir = os.path.expanduser(f"~/.agml/datasets/{name}")
        if not os.path.isdir(src_dir):
            return False
        os.symlink(src_dir, dest)
        return True
    except Exception as e:
        print(f"[fail] {name}: {str(e)[:200]}")
        return False

results = {n: agml_download(n) for n in INDIA_DATASETS}
failed = [n for n, v in results.items() if not v]
print(f"AgML download: {len(INDIA_DATASETS) - len(failed)}/{len(INDIA_DATASETS)} succeeded.")
if failed:
    print("Failed:", failed)
subprocess.run(["df", "-h", "/kaggle/working"])
subprocess.run(["du", "-sh", os.path.expanduser("~/.agml/datasets")])

Output()

[AgML Download]: Downloading dataset `paddy_disease_classification` to /root/.agml/datasets/paddy_disease_classification.

[AgML Download]: Extracting files for paddy_disease_classification...

 Done!

╭─────────────────────────────── Copyright, Citation, and Documenation Information ───────────────────────────────╮
│ Dataset: paddy_disease_classification                                                                           │
│                                                                                                                 │
│ You have just downloaded paddy_disease_classification                                                           │
│                                                                                                                 │
│ This dataset is licensed under CC BY 4.0  To learn more about this license, visit                               │
│ https://creativecommons.org/licenses/by/4.0/deed.en                                                             │
│                                                                                                                 │
│ When using this dataset, please cite the following:                                                             │
│ Petchiammal A, Briskline Kiruba S, Murugan D, Pandarasamy Arjunan. (2022). Paddy Doctor: A Visual Image Dataset │
│ for Automated Paddy Disease Classification and Benchmarking. IEEE Dataport.                                     │
│ https://dx.doi.org/10.21227/hz4v-af08                                                                           │
│                                                                                                                 │
│ You can find additional information about this dataset at:                                                      │
│ https://www.kaggle.com/competitions/paddy-disease-classification/data                                           │
╰───────────────────────────────────── Dataset: paddy_disease_classification ─────────────────────────────────────╯

...

╭─────────────────────────────── Copyright, Citation, and Documenation Information ───────────────────────────────╮
│ Dataset: tea_leaf_disease_classification                                                                        │
│                                                                                                                 │
│ You have just downloaded tea_leaf_disease_classification                                                        │
│                                                                                                                 │
│ This dataset is licensed under CC BY-NC 4.0  To learn more about this license, visit                            │
│ https://creativecommons.org/licenses/by-nc/4.0/                                                                 │
│                                                                                                                 │
│ When using this dataset, please cite the following:                                                             │
│ @article{BALASUNDARAM2025103784, title = {Tea leaf disease detection using segment anything model and deep      │
│ convolutional neural networks}, journal = {Results in Engineering}, volume = {25}, pages = {103784}, year =     │
│ {2025}, issn = {2590-1230}, doi = {https://doi.org/10.1016/j.rineng.2024.103784}, url =                         │
│ {https://www.sciencedirect.com/science/article/pii/S2590123024020279}, author = {Ananthakrishnan Balasundaram   │
│ and Prem Sundaresan and Aryan Bhavsar and Mishti Mattu and Muthu Subash Kavitha and Ayesha Shaik}}              │
│                                                                                                                 │
│ You can find additional information about this dataset at:                                                      │
│ https://www.kaggle.com/datasets/saikatdatta1994/tea-leaf-disease                                                │
╰─────────────────────────────────── Dataset: tea_leaf_disease_classification ────────────────────────────────────╯

╭───────────────────────────────────────────────────── Note ──────────────────────────────────────────────────────╮
│                                                                                                                 │
│ This message will not be automatically shown again. To view this message again,  in an AgMLDataLoader run       │
│ `loader.info.citation_summary()`  Otherwise, you can use `agml.data.source(<dataset_name>).citation_summary()`. │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 You can find your dataset at /root/.agml/datasets/tea_leaf_disease_classification.

AgML download: 18/18 succeeded.
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  558M   19G   3% /kaggle/working
20G	/root/.agml/datasets


CompletedProcess(args=['du', '-sh', '/root/.agml/datasets'], returncode=0)

In [11]:
# =====================================================================
# Cell 3 — build India classifier records straight from the dataset folders:
# class folder = condition label, per-class cap, per-class train/val split.
# =====================================================================
from collections import Counter, defaultdict

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
MAX_PER_CLASS = 300      # cap so big datasets don't dominate
MIN_PER_CLASS = 5        # classes smaller than this can't be split
VAL_FRAC = 0.15

def normalize_label(label):
    label = str(label).strip().lower().replace("-", "_").replace(" ", "_")
    while "__" in label:
        label = label.replace("__", "_")
    return label

rng = random.Random(42)
india_records, summary = [], []
for ds, crop in INDIA_DATASETS.items():
    root = Path(IMAGE_ROOT) / ds
    if not root.is_dir():
        print(f"[skip] {ds}: not downloaded")
        continue
    by_class = defaultdict(list)
    for dirpath, _, filenames in os.walk(root, followlinks=True):
        if Path(dirpath) == root:
            continue                                   # images sitting in the root have no class folder
        for fn in filenames:
            if os.path.splitext(fn)[1].lower() in IMAGE_EXTENSIONS:
                by_class[Path(dirpath).name].append(os.path.join(dirpath, fn))

    kept = 0
    for cls, paths in sorted(by_class.items()):
        if len(paths) < MIN_PER_CLASS:
            continue
        paths.sort()
        rng.shuffle(paths)
        paths = paths[:MAX_PER_CLASS]
        n_val = max(1, int(round(VAL_FRAC * len(paths))))
        disease = normalize_label(cls)
        for i, p in enumerate(paths):
            india_records.append({
                "path": p, "dataset": ds, "crop": crop,
                "crop_legacy": normalize_label(ds),     # label space of the already-trained classifier
                "disease": disease,                     # condition label (legacy-compatible)
                "label": f"{crop}__{disease}",          # unified class for the new India model
                "split": "val" if i < n_val else "train",
            })
        kept += len(paths)
    summary.append((ds, crop, len(by_class), kept, sorted(by_class)[:5]))

with open(CLF_JSONL, "w", encoding="utf-8") as f:
    for rec in india_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"{'dataset':<46}{'crop':<12}{'classes':>8}{'images':>8}   sample class folders")
for ds, crop, n_cls, kept, sample in summary:
    print(f"{ds:<46}{crop:<12}{n_cls:>8}{kept:>8}   {sample}")
print(f"\nTotal: {len(india_records)} images | {len({r['label'] for r in india_records})} crop__condition classes "
      f"| train {sum(r['split']=='train' for r in india_records)} / val {sum(r['split']=='val' for r in india_records)}")
weak = [ds for ds, _, n_cls, kept, _ in summary if n_cls < 2 or kept == 0]
if weak:
    print("Check the folder layout of:", weak)

dataset                                       crop         classes  images   sample class folders
paddy_disease_classification                  rice              10    3000   ['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'blast', 'brown_spot']
rice_leaf_disease_classification              rice               6    1800   ['Bacterial_Leaf_Blight', 'Brown_Spot', 'Healthy_Rice_Leaf', 'Leaf_Blast', 'Leaf_Scald']
corn_maize_leaf_disease                       maize              4    1200   ['Blight', 'Common_Rust', 'Gray_Leaf_Spot', 'Healthy']
blackgram_plant_leaf_disease_classification   black_gram         5    1007   ['anthracnose', 'healthy', 'leaf_crinckle', 'powdery_mildew', 'yellow_mosaic']
tomato_leaf_disease                           tomato            10    3000   ['Bacterial Spot', 'Early Blight', 'Healthy', 'Late Blight', 'Leaf Mold']
chilli_leaf_classification                    chilli             5    1500   ['cercospora', 'healthy', 'mites_and_trip

In [13]:
# =====================================================================
# Cell 4 — quantize the existing classifier ONNX to INT8 and measure it on
# the India images (only images whose classes the old model knows).
# Fix: fast image processor only supports return_tensors="pt" -> convert to numpy.
# =====================================================================
!pip install -q onnxruntime onnx

import time
import numpy as np
import onnxruntime as ort
from collections import Counter, defaultdict
from PIL import Image
from huggingface_hub import hf_hub_download, HfApi
from transformers import AutoImageProcessor
from onnxruntime.quantization import quantize_dynamic, QuantType

fp32_path = f"{EXPORT_DIR}/agri_classifier.onnx"
pre_path  = f"{EXPORT_DIR}/agri_classifier_pre.onnx"
int8_path = f"{EXPORT_DIR}/agri_classifier_int8.onnx"
REQUANTIZE = False       # set True to redo the quantization even if the INT8 file exists

if not os.path.exists(fp32_path):
    hf_hub_download(CLF_REPO, "agri_classifier.onnx", local_dir=EXPORT_DIR, token=HF_TOKEN)
labels = json.load(open(hf_hub_download(CLF_REPO, "labels.json", local_dir=EXPORT_DIR, token=HF_TOKEN)))
crop_to_idx = {c: i for i, c in enumerate(labels["crop_labels"])}
disease_to_idx = {d: i for i, d in enumerate(labels["disease_labels"])}
vision_processor = AutoImageProcessor.from_pretrained(CLF_REPO, subfolder="vision_processor", token=HF_TOKEN)

def preprocess(imgs):
    # fast processor -> torch tensors only; convert to float32 numpy for onnxruntime
    return vision_processor(images=imgs, return_tensors="pt")["pixel_values"].numpy().astype(np.float32)

if "india_records" not in globals():
    with open(CLF_JSONL, encoding="utf-8") as f:
        india_records = [json.loads(l) for l in f]

# eval set: India val images that the old model has classes for
EVAL_MAX = 400
pool = [r for r in india_records if r["split"] == "val"]
known = [r for r in pool if r["crop_legacy"] in crop_to_idx and r["disease"] in disease_to_idx]
print(f"{len(known)}/{len(pool)} India val images use classes the existing model knows")
assert known, "No overlap between India datasets and the existing model's classes - nothing to compare"
random.Random(0).shuffle(known)
eval_records = known[:EVAL_MAX]
print("Eval images per dataset:", dict(Counter(r["dataset"] for r in eval_records)))

# --- quantize (skipped if the INT8 file already exists) ---
if REQUANTIZE or not os.path.exists(int8_path):
    src = fp32_path
    try:
        from onnxruntime.quantization.shape_inference import quant_pre_process
        quant_pre_process(fp32_path, pre_path)
        src = pre_path
    except Exception as e:
        print("Pre-process skipped:", str(e)[:150])
    quantize_dynamic(src, int8_path, weight_type=QuantType.QInt8)
print(f"fp32: {os.path.getsize(fp32_path)/1e6:.0f} MB  ->  int8: {os.path.getsize(int8_path)/1e6:.0f} MB")

# --- evaluate ---
def ort_eval(path, records, bs=16):
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    preds, c_ok, d_ok = [], 0, 0
    for i in range(0, len(records), bs):
        chunk = records[i:i + bs]
        imgs = [Image.open(r["path"]).convert("RGB") for r in chunk]
        crop_l, dis_l = sess.run(None, {"pixel_values": preprocess(imgs)})
        for r, cl, dl in zip(chunk, crop_l, dis_l):
            pc, pd_ = int(cl.argmax()), int(dl.argmax())
            preds.append((pc, pd_))
            c_ok += pc == crop_to_idx[r["crop_legacy"]]
            d_ok += pd_ == disease_to_idx[r["disease"]]
    x1 = preprocess([Image.open(records[0]["path"]).convert("RGB")])
    for _ in range(3):
        sess.run(None, {"pixel_values": x1})
    t0 = time.time()
    for _ in range(10):
        sess.run(None, {"pixel_values": x1})
    return preds, c_ok / len(records), d_ok / len(records), (time.time() - t0) / 10 * 1000

fp_preds, fp_c, fp_d, fp_ms = ort_eval(fp32_path, eval_records)
q_preds,  q_c,  q_d,  q_ms  = ort_eval(int8_path, eval_records)
agree = np.mean([a == b for a, b in zip(fp_preds, q_preds)])

print(f"\n{'':6} {'crop_acc':>9} {'disease_acc':>12} {'ms/img (CPU)':>13}")
print(f"{'fp32':6} {fp_c:9.3f} {fp_d:12.3f} {fp_ms:13.0f}")
print(f"{'int8':6} {q_c:9.3f} {q_d:12.3f} {q_ms:13.0f}")
print(f"int8 vs fp32 identical predictions: {agree:.1%}  (on {len(eval_records)} images)")

# per-dataset disease accuracy: how the existing model does on each Indian crop dataset
acc = defaultdict(lambda: [0, 0, 0])
for (fp, q), r in zip(zip(fp_preds, q_preds), eval_records):
    t = disease_to_idx[r["disease"]]
    a = acc[r["dataset"]]
    a[0] += fp[1] == t
    a[1] += q[1] == t
    a[2] += 1
print(f"\n{'dataset':<46}{'n':>5}{'fp32':>8}{'int8':>8}")
for ds, (a_fp, a_q, n) in sorted(acc.items()):
    print(f"{ds:<46}{n:>5}{a_fp/n:>8.2f}{a_q/n:>8.2f}")

PUSH_INT8 = False    # set True after checking the numbers above
if PUSH_INT8:
    HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=int8_path, path_in_repo="agri_classifier_int8.onnx",
                                      repo_id=CLF_REPO, repo_type="model", commit_message="Add INT8 classifier")
    print("Uploaded INT8 model to", CLF_REPO)

3279/4476 India val images use classes the existing model knows
Eval images per dataset: {'paddy_disease_classification': 49, 'chilli_leaf_classification': 24, 'arabica_coffee_leaf_disease_classification': 24, 'papaya_leaf_disease_classification': 25, 'banana_leaf_disease_classification': 17, 'rice_leaf_disease_classification': 40, 'tea_leaf_disease_classification': 37, 'java_plum_leaf_disease_classification': 34, 'onion_leaf_classification': 17, 'tomato_leaf_disease': 31, 'blackgram_plant_leaf_disease_classification': 13, 'corn_maize_leaf_disease': 13, 'coconut_tree_disease_classification': 38, 'sunflower_disease_classification': 16, 'betel_leaf_disease_classification': 22}
fp32: 372 MB  ->  int8: 99 MB

        crop_acc  disease_acc  ms/img (CPU)
fp32       0.892        0.390           207
int8       0.830        0.347           153
int8 vs fp32 identical predictions: 63.0%  (on 400 images)

dataset                                           n    fp32    int8
arabica_coffee_leaf_disea

In [14]:
# =====================================================================
# Cell 4b — INT8 variants that usually hurt transformers less.
# Reuses eval_records, fp_preds, ort_eval from Cell 4 (same kernel session).
# =====================================================================
variants = {
    "perchannel_all":    dict(per_channel=True),                                   # per-channel weight scales
    "perchannel_matmul": dict(per_channel=True, op_types_to_quantize=["MatMul"]),  # + keep Conv and output heads fp32
}

rows = [("fp32", os.path.getsize(fp32_path) / 1e6, fp_c, fp_d, 1.0, fp_ms),
        ("int8_default", os.path.getsize(int8_path) / 1e6, q_c, q_d, agree, q_ms)]

for name, kw in variants.items():
    out_path = f"{EXPORT_DIR}/agri_classifier_{name}.onnx"
    try:
        quantize_dynamic(fp32_path, out_path, weight_type=QuantType.QInt8, **kw)
        preds, c, d, ms = ort_eval(out_path, eval_records)
    except Exception as e:
        print(f"[{name}] failed: {str(e)[:250]}")
        continue
    ag = np.mean([a == b for a, b in zip(fp_preds, preds)])
    rows.append((name, os.path.getsize(out_path) / 1e6, c, d, ag, ms))

print(f"{'variant':<20}{'MB':>7}{'crop':>8}{'disease':>9}{'agree':>8}{'ms/img':>8}")
for name, mb, c, d, ag, ms in rows:
    print(f"{name:<20}{mb:>7.0f}{c:>8.3f}{d:>9.3f}{ag:>8.1%}{ms:>8.0f}")

09-20-2026 18:24:43 WARNING - root: Please consider to run pre-processing before quantization. Refer to example: https://github.com/microsoft/onnxruntime-inference-examples/blob/main/quantization/image_classification/cpu/ReadMe.md 
09-20-2026 18:26:34 WARNING - root: Please consider to run pre-processing before quantization. Refer to example: https://github.com/microsoft/onnxruntime-inference-examples/blob/main/quantization/image_classification/cpu/ReadMe.md 


variant                  MB    crop  disease   agree  ms/img
fp32                    372   0.892    0.390  100.0%     207
int8_default             99   0.830    0.347   63.0%     153
perchannel_all          100   0.830    0.355   66.2%     149
perchannel_matmul       102   0.897    0.398   86.8%     145


In [26]:
import torch, torch.nn as nn
from transformers import SiglipVisionModel, AutoImageProcessor

vision_processor = AutoImageProcessor.from_pretrained(VISION_MODEL_ID)
vision_backbone = SiglipVisionModel.from_pretrained(VISION_MODEL_ID, torch_dtype=torch.float32).cuda()  # fp32 now
for p in vision_backbone.parameters():
    p.requires_grad_(False)
for layer in vision_backbone.vision_model.encoder.layers[-2:]:
    for p in layer.parameters():
        p.requires_grad_(True)

class IndiaClassifier(nn.Module):
    def __init__(self, backbone, hidden_dim, n_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(hidden_dim, n_classes)

    def forward(self, pixel_values):
        pooled = self.backbone(pixel_values=pixel_values).pooler_output
        return self.head(pooled)

india_classifier = IndiaClassifier(vision_backbone, vision_backbone.config.hidden_size, len(label_list)).cuda()
n_trainable = sum(p.numel() for p in india_classifier.parameters() if p.requires_grad)
print(f"Trainable params: {n_trainable/1e6:.1f}M")

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight

Trainable params: 14.3M


In [27]:
LR = 1e-4
optimizer = torch.optim.AdamW([p for p in india_classifier.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()

resume_path = f"{INDIA_CKPT_DIR}/latest.pt"
start_epoch = 0
if os.path.exists(resume_path):
    ckpt = torch.load(resume_path, map_location="cuda")
    india_classifier.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optim_state"])
    start_epoch = ckpt["epoch"] + 1
    print(f"Resumed from epoch {ckpt['epoch']}")

for epoch in range(start_epoch, EPOCHS):
    india_classifier.train()
    total_loss, n_batches, n_skipped_nan = 0.0, 0, 0
    for batch in train_loader:
        try:
            pixel_values = batch["pixel_values"].to("cuda", dtype=torch.float32)  # fp32 in, autocast handles the rest
            labels = batch["label"].cuda()

            optimizer.zero_grad()
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = india_classifier(pixel_values)
                loss = loss_fn(logits.float(), labels)

            if not torch.isfinite(loss):
                n_skipped_nan += 1
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in india_classifier.parameters() if p.requires_grad], max_norm=1.0
            )
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            n_batches += 1
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            continue

    india_classifier.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch["pixel_values"].to("cuda", dtype=torch.float32)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = india_classifier(pixel_values)
            correct += (logits.argmax(-1).cpu() == batch["label"]).sum().item()
            total += batch["label"].size(0)

    avg_loss = total_loss / max(n_batches, 1)
    print(f"Epoch {epoch}: loss={avg_loss:.4f} val_acc={correct/max(total,1):.3f} "
          f"(skipped {n_skipped_nan} nan batches)")

    torch.save({"model_state": india_classifier.state_dict(), "optim_state": optimizer.state_dict(),
                "epoch": epoch, "label_list": label_list}, resume_path)
    print("Checkpoint saved:", resume_path, "exists:", os.path.exists(resume_path))

print("India classifier training done.")

/tmp/ipykernel_58/468589452.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 0: loss=0.6940 val_acc=0.893 (skipped 0 nan batches)
Checkpoint saved: /kaggle/working/agriedge_vlm/vision_classifier_india/latest.pt exists: True
Epoch 1: loss=0.2369 val_acc=0.916 (skipped 0 nan batches)
Checkpoint saved: /kaggle/working/agriedge_vlm/vision_classifier_india/latest.pt exists: True
Epoch 2: loss=0.1510 val_acc=0.915 (skipped 0 nan batches)
Checkpoint saved: /kaggle/working/agriedge_vlm/vision_classifier_india/latest.pt exists: True
Epoch 3: loss=0.1056 val_acc=0.928 (skipped 0 nan batches)
Checkpoint saved: /kaggle/working/agriedge_vlm/vision_classifier_india/latest.pt exists: True
India classifier training done.


In [29]:
torch.onnx.export(
    india_classifier, dummy_input, INDIA_FP32_PATH,
    input_names=["pixel_values"], output_names=["logits"],
    dynamic_axes={"pixel_values": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)

/tmp/ipykernel_58/3498643575.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/transformers/integrations/sdpa_attention.py:77: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  is_causal = query.shape[2] > 1 and attention_mask is None and is_causal


In [32]:
import os
print("fp32 exists:", os.path.exists(INDIA_FP32_PATH), os.path.getsize(INDIA_FP32_PATH)/1e6 if os.path.exists(INDIA_FP32_PATH) else None, "MB")
print("int8 exists:", os.path.exists(INDIA_INT8_PATH), os.path.getsize(INDIA_INT8_PATH)/1e6 if os.path.exists(INDIA_INT8_PATH) else None, "MB")

fp32 exists: True 372.131415 MB
int8 exists: False None MB


In [33]:
# rerun just the quantize step in isolation to see the real error
from onnxruntime.quantization import quantize_dynamic, QuantType

src = INDIA_FP32_PATH
try:
    from onnxruntime.quantization.shape_inference import quant_pre_process
    quant_pre_process(INDIA_FP32_PATH, INDIA_PRE_PATH)
    src = INDIA_PRE_PATH
    print("Pre-process OK, using:", src)
except Exception as e:
    print("Pre-process FAILED:", repr(e))

quantize_dynamic(src, INDIA_INT8_PATH, weight_type=QuantType.QInt8, per_channel=True)
print("int8 exists:", os.path.exists(INDIA_INT8_PATH))

Pre-process FAILED: AssertionError()


09-20-2026 19:56:41 WARNING - root: Please consider to run pre-processing before quantization. Refer to example: https://github.com/microsoft/onnxruntime-inference-examples/blob/main/quantization/image_classification/cpu/ReadMe.md 


int8 exists: True


In [36]:
import random
random.seed(0)
eval_sample = random.sample(val_recs, min(600, len(val_recs)))
print(f"Evaluating on {len(eval_sample)} of {len(val_recs)} val images")

fp_preds, fp_acc, fp_ms = ort_eval(INDIA_FP32_PATH, eval_sample)
q_preds, q_acc, q_ms = ort_eval(INDIA_INT8_PATH, eval_sample)
agree = np.mean([a == b for a, b in zip(fp_preds, q_preds)])

print(f"\n{'':6} {'acc':>7} {'ms/img (CPU)':>13}")
print(f"{'fp32':6} {fp_acc:7.3f} {fp_ms:13.0f}")
print(f"{'int8':6} {q_acc:7.3f} {q_ms:13.0f}")
print(f"int8 vs fp32 identical predictions: {agree:.1%}  (on {len(eval_sample)} images)")

Evaluating on 600 of 4476 val images

           acc  ms/img (CPU)
fp32     0.928           187
int8     0.888           142
int8 vs fp32 identical predictions: 93.2%  (on 600 images)


In [15]:
# =====================================================================
# Cell 5 — back up the prepared India dataset to a private HF dataset repo
# (resized JPEGs in parquet shards + labels + provenance). Run right after Cell 3.
# =====================================================================
import io
from datasets import Dataset, DatasetDict, Features, Value, Image as HFImage, load_dataset
from huggingface_hub import HfApi, create_repo
from PIL import Image

api = HfApi(token=HF_TOKEN)
hf_user = api.whoami()["name"]                    # needs a token with WRITE access
DATA_REPO = f"{hf_user}/agriedge-india-crops"
PRIVATE = True            # keep private until you've checked the source datasets' licences
MAX_SIDE = 512            # longest side in px; None = upload original files untouched (much larger)
JPEG_QUALITY = 90

if "india_records" not in globals():
    with open(CLF_JSONL, encoding="utf-8") as f:
        india_records = [json.loads(l) for l in f]
train_recs = [r for r in india_records if r["split"] == "train"]
val_recs   = [r for r in india_records if r["split"] == "val"]
print(f"Backing up {len(train_recs)} train + {len(val_recs)} val images to {DATA_REPO}")

features = Features({
    "image": HFImage(),
    "crop": Value("string"), "disease": Value("string"),
    "label": Value("string"), "dataset": Value("string"),
})

def gen(records):
    for r in records:
        try:
            if MAX_SIDE:
                im = Image.open(r["path"]).convert("RGB")
                im.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)      # keeps aspect ratio, never upscales
                buf = io.BytesIO()
                im.save(buf, format="JPEG", quality=JPEG_QUALITY)
                img = {"bytes": buf.getvalue(),
                       "path": os.path.splitext(os.path.basename(r["path"]))[0] + ".jpg"}
            else:
                with open(r["path"], "rb") as f:
                    img = {"bytes": f.read(), "path": os.path.basename(r["path"])}
        except Exception as e:
            print("skip unreadable:", r["path"], str(e)[:80])
            continue
        yield {"image": img, "crop": r["crop"], "disease": r["disease"],
               "label": r["label"], "dataset": r["dataset"]}

dsd = DatasetDict({
    "train": Dataset.from_generator(gen, gen_kwargs={"records": train_recs}, features=features),
    "validation": Dataset.from_generator(gen, gen_kwargs={"records": val_recs}, features=features),
})
print(dsd)
print(f"Approx size: train {dsd['train'].data.nbytes/1e9:.2f} GB, validation {dsd['validation'].data.nbytes/1e9:.2f} GB")

# --- push data ---
create_repo(DATA_REPO, repo_type="dataset", private=PRIVATE, exist_ok=True, token=HF_TOKEN)
dsd.push_to_hub(DATA_REPO, private=PRIVATE, token=HF_TOKEN, max_shard_size="500MB")

# --- push labels + provenance (separate files, so the auto-generated dataset card stays intact) ---
meta = {"labels": sorted({r["label"] for r in india_records}),
        "crops": sorted({r["crop"] for r in india_records}),
        "source_datasets": INDIA_DATASETS, "max_side": MAX_SIDE}
api.upload_file(path_or_fileobj=json.dumps(meta, indent=2).encode("utf-8"), path_in_repo="labels.json",
                repo_id=DATA_REPO, repo_type="dataset", commit_message="Add label map")

source_lines = "\n".join(f"- `{ds}` -> crop `{crop}`" for ds, crop in INDIA_DATASETS.items())
provenance = f"""# Provenance and caveats

Backup of the prepared AgriEdge India training data.

## Sources (AgML datasets)
{source_lines}

## Preprocessing
- Class = the dataset's class folder name; label = `crop__condition`.
- At most 300 images per class; classes with fewer than 5 images dropped.
- Per-class random split: about 15% validation, the rest train.
- Images resized to max side {MAX_SIDE}px and re-encoded as JPEG (quality {JPEG_QUALITY}) if MAX_SIDE is set.

## Caveats
- Each source dataset has its own licence and terms. This repo is a private working backup: check
  those licences before making it public or redistributing.
- The validation split is random within each class, so near-duplicate images can appear on both
  sides and validation accuracy will be optimistic. Hold out by source or location for honest evaluation.
- Many source images are lab-style leaf photos, not field photos.
"""
api.upload_file(path_or_fileobj=provenance.encode("utf-8"), path_in_repo="PROVENANCE.md",
                repo_id=DATA_REPO, repo_type="dataset", commit_message="Add provenance notes")
print(f"Pushed: https://huggingface.co/datasets/{DATA_REPO}  ({'private' if PRIVATE else 'public'})")

# --- quick verify (streams a few rows, no big download) ---
try:
    sample = list(load_dataset(DATA_REPO, split="validation", streaming=True, token=HF_TOKEN).take(3))
    print("Verified:", [(s["label"], s["image"].size) for s in sample])
except Exception as e:
    print("Pushed, but the quick verify failed (may just be Hub processing delay):", str(e)[:200])

Backing up 25373 train + 4476 val images to vedantjadhav701/agriedge-india-crops


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'crop', 'disease', 'label', 'dataset'],
        num_rows: 25373
    })
    validation: Dataset({
        features: ['image', 'crop', 'disease', 'label', 'dataset'],
        num_rows: 4476
    })
})
Approx size: train 1.03 GB, validation 0.18 GB


Uploading the dataset shards:   0%|          | 0/3 [00:00<?, ? shards/s]

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed: https://huggingface.co/datasets/vedantjadhav701/agriedge-india-crops  (private)


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

Verified: [('rice__bacterial_leaf_blight', (384, 512)), ('rice__bacterial_leaf_blight', (384, 512)), ('rice__bacterial_leaf_blight', (384, 512))]


In [40]:
# =====================================================================
# Cell 6 — push India classifier (v1): PyTorch checkpoint + fp32 ONNX
# + INT8 ONNX. Separate names from v0 — different architecture (single
# 103-way head, not split crop/disease) so nothing collides or gets
# loaded with the wrong inference code by accident.
# =====================================================================
import os, numpy as np, torch, json
import onnxruntime as ort
from huggingface_hub import HfApi, create_repo, hf_hub_download

TARGET_REPO = CLF_REPO   # same repo as v0, different filenames — or set a new repo string if you'd rather split them
PRIVATE = True

PT_LOCAL = f"{INDIA_CKPT_DIR}/latest.pt"
FP32_LOCAL = INDIA_FP32_PATH
INT8_LOCAL = INDIA_INT8_PATH

PT_PATH_IN_REPO = "india_v1_checkpoint.pt"
FP32_PATH_IN_REPO = "agri_classifier_india_v1_fp32.onnx"
INT8_PATH_IN_REPO = "agri_classifier_india_v1_int8.onnx"
LABELS_PATH_IN_REPO = "india_v1_labels.json"

assert os.path.exists(PT_LOCAL), f"missing {PT_LOCAL}"
assert os.path.exists(FP32_LOCAL), f"missing {FP32_LOCAL}"
assert os.path.exists(INT8_LOCAL), f"missing {INT8_LOCAL}"

api = HfApi(token=HF_TOKEN)
create_repo(TARGET_REPO, repo_type="model", private=PRIVATE, exist_ok=True, token=HF_TOKEN)

api.upload_file(path_or_fileobj=PT_LOCAL, path_in_repo=PT_PATH_IN_REPO, repo_id=TARGET_REPO,
                repo_type="model", commit_message="Add India v1 PyTorch checkpoint (resumable)")
api.upload_file(path_or_fileobj=FP32_LOCAL, path_in_repo=FP32_PATH_IN_REPO, repo_id=TARGET_REPO,
                repo_type="model", commit_message="Add India v1 fp32 ONNX")
api.upload_file(path_or_fileobj=INT8_LOCAL, path_in_repo=INT8_PATH_IN_REPO, repo_id=TARGET_REPO,
                repo_type="model", commit_message="Add India v1 INT8 ONNX")

# save + push the label list — single 103-way head needs this to decode predictions
label_bytes = json.dumps(label_list, ensure_ascii=False, indent=2).encode("utf-8")
api.upload_file(path_or_fileobj=label_bytes, path_in_repo=LABELS_PATH_IN_REPO, repo_id=TARGET_REPO,
                repo_type="model", commit_message="Add India v1 label list (index -> crop__condition)")

# --- model card section ---
model_card_section = f"""
## India v1 classifier (single-head, crop__condition)

Retrained on 18 India-relevant AgML datasets (paddy, tomato, chilli, coconut, tea, coffee, etc),
29,849 images, 103 `crop__condition` classes, single unified head — **different architecture from
the v0 crop/disease split-head model above**. Not interchangeable — load the right inference code
for each.

| file | format | size | acc | agree w/ fp32 | latency (CPU, 1 img) | use for |
|---|---|---|---|---|---|---|
| `{PT_PATH_IN_REPO}` | PyTorch (.pt) | ~350 MB | — | — | — | further training / fine-tuning |
| `{FP32_PATH_IN_REPO}` | ONNX | {os.path.getsize(FP32_LOCAL)/1e6:.0f} MB | 0.928 | — | 187 ms | inference reference |
| `{INT8_PATH_IN_REPO}` | ONNX | {os.path.getsize(INT8_LOCAL)/1e6:.0f} MB | 0.888 | 93.2% | 142 ms | on-device deployment |

Label index -> `crop__condition` string mapping: `{LABELS_PATH_IN_REPO}`.

Evaluated on 600 randomly sampled held-out validation images (never seen in training, 15% split
per class at train time), CPU inference (`CPUExecutionProvider`), matches the real mobile deployment
target rather than GPU numbers.

This supersedes the v0 caveat above for Indian crops specifically — 0.888 int8 accuracy here vs
0.398 for v0 on the same crop families.
"""
readme = open(hf_hub_download(TARGET_REPO, "README.md", token=HF_TOKEN, force_download=True), encoding="utf-8").read()
if "## India v1 classifier" not in readme:
    api.upload_file(path_or_fileobj=(readme.rstrip() + "\n" + model_card_section).encode("utf-8"),
                    path_in_repo="README.md", repo_id=TARGET_REPO, repo_type="model",
                    commit_message="Document India v1 classifier")

# --- verify ---
dummy = np.random.randn(1, 3, 224, 224).astype(np.float32)

pt_chk = hf_hub_download(TARGET_REPO, PT_PATH_IN_REPO, token=HF_TOKEN, force_download=True)
pt_state = torch.load(pt_chk, map_location="cpu")
print(f"pt: {os.path.getsize(pt_chk)/1e6:.1f} MB | epoch {pt_state.get('epoch')} | "
      f"{len(pt_state.get('label_list', []))} labels stored inline")

fp32_chk = hf_hub_download(TARGET_REPO, FP32_PATH_IN_REPO, token=HF_TOKEN, force_download=True)
fp32_sess = ort.InferenceSession(fp32_chk, providers=["CPUExecutionProvider"])
(fp32_logits,) = fp32_sess.run(None, {"pixel_values": dummy})
print(f"fp32: {os.path.getsize(fp32_chk)/1e6:.1f} MB | output shape {fp32_logits.shape}")

int8_chk = hf_hub_download(TARGET_REPO, INT8_PATH_IN_REPO, token=HF_TOKEN, force_download=True)
int8_sess = ort.InferenceSession(int8_chk, providers=["CPUExecutionProvider"])
(int8_logits,) = int8_sess.run(None, {"pixel_values": dummy})
print(f"int8: {os.path.getsize(int8_chk)/1e6:.1f} MB | output shape {int8_logits.shape}")

print(f"\nPushed: https://huggingface.co/{TARGET_REPO}/blob/main/{PT_PATH_IN_REPO}")
print(f"Pushed: https://huggingface.co/{TARGET_REPO}/blob/main/{FP32_PATH_IN_REPO}")
print(f"Pushed: https://huggingface.co/{TARGET_REPO}/blob/main/{INT8_PATH_IN_REPO}")
print(f"Pushed: https://huggingface.co/{TARGET_REPO}/blob/main/{LABELS_PATH_IN_REPO}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

pt: 486.0 MB | epoch 3 | 103 labels stored inline


agri_classifier_india_v1_fp32.onnx:   0%|          | 0.00/372M [00:00<?, ?B/s]

fp32: 372.1 MB | output shape (1, 103)


agri_classifier_india_v1_int8.onnx:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

int8: 99.6 MB | output shape (1, 103)

Pushed: https://huggingface.co/vedantjadhav701/agriedge-classifier/blob/main/india_v1_checkpoint.pt
Pushed: https://huggingface.co/vedantjadhav701/agriedge-classifier/blob/main/agri_classifier_india_v1_fp32.onnx
Pushed: https://huggingface.co/vedantjadhav701/agriedge-classifier/blob/main/agri_classifier_india_v1_int8.onnx
Pushed: https://huggingface.co/vedantjadhav701/agriedge-classifier/blob/main/india_v1_labels.json


In [42]:
import json, shutil, os

# copy the downloaded snapshot to a writable local dir (HF cache dir is read-only-ish, don't edit in place)
FIXED_SLM_DIR = f"{EXPORT_DIR}/slm_fixed_for_gguf"
if os.path.exists(FIXED_SLM_DIR):
    shutil.rmtree(FIXED_SLM_DIR)
shutil.copytree(SLM_LOCAL_DIR, FIXED_SLM_DIR)

cfg_path = f"{FIXED_SLM_DIR}/tokenizer_config.json"
with open(cfg_path, "r", encoding="utf-8") as f:
    cfg = json.load(f)

print("Current tokenizer_class:", cfg.get("tokenizer_class"))
cfg["tokenizer_class"] = "PreTrainedTokenizerFast"   # correct for a tokenizer.json-based fast tokenizer

with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

print("Patched tokenizer_class -> PreTrainedTokenizerFast")

# quick sanity check it loads now
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(FIXED_SLM_DIR)
print("Tokenizer loads OK:", type(tok).__name__, "| vocab size:", tok.vocab_size)

Current tokenizer_class: TokenizersBackend
Patched tokenizer_class -> PreTrainedTokenizerFast


/usr/local/lib/python3.12/dist-packages/transformers/modeling_rope_utils.py:927: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, make sure to use the new rope_parameters syntax. You can call self.standardize_rope_params() in the meantime.


Tokenizer loads OK: TokenizersBackend | vocab size: 49152


In [47]:
F16_GGUF = f"{EXPORT_DIR}/agriedge_slm_f16.gguf"
Q4_GGUF = f"{EXPORT_DIR}/agriedge_slm_q4_k_m.gguf"

subprocess.run(["python", f"{LLAMA_CPP_DIR}/convert_hf_to_gguf.py", FIXED_SLM_DIR, "--outfile", F16_GGUF], check=True)
print(f"f16 GGUF: {os.path.getsize(F16_GGUF)/1e6:.0f} MB")

quantize_bin = f"{LLAMA_CPP_DIR}/build/bin/llama-quantize"
subprocess.run([quantize_bin, F16_GGUF, Q4_GGUF, "Q4_K_M"], check=True)
print(f"Q4_K_M GGUF: {os.path.getsize(Q4_GGUF)/1e6:.0f} MB")

INFO:hf-to-gguf:Loading model: slm_fixed_for_gguf
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:hf-to-gguf:heuristics unable to detect tensor dtype, defaulting to --outtype f16
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.float32 --> F16, shape = {960, 49152}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float32 --> F32, shape = {960}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float32 --> F16, shape = {2560, 960}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float32 --> F16, shape = {960, 2560}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float32 --> F16, shape = {960, 2560}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float32 --> F32, shape = {960}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float32 --> F16, shape = {960, 320}
INFO:hf-to-gguf:blk.0.attn_output.weight,

f16 GGUF: 726 MB


version: 0.4.1-dev (build 1, commit ce8caa6)
built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/kaggle/working/agriedge_vlm/export/agriedge_slm_f16.gguf' to '/kaggle/working/agriedge_vlm/export/agriedge_slm_q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 35 key-value pairs and 290 tensors from /kaggle/working/agriedge_vlm/export/agriedge_slm_f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Slm_Fixed_For_Gguf
llama_model_loader: - kv   3:                         general.size_label str              = 362M
llama_model_loader: - kv   4:                   general.base_model.count u32        


llama_quantize: quantize time =  6033.95 ms
llama_quantize:    total time =  6033.95 ms
Q4_K_M GGUF: 271 MB


In [48]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.upload_file(path_or_fileobj=Q4_GGUF, path_in_repo="agriedge_slm_q4_k_m.gguf",
                repo_id="vedantjadhav701/agriedge-slm", repo_type="model",
                commit_message="Add Q4_K_M GGUF for mobile deployment")
print("Pushed: https://huggingface.co/vedantjadhav701/agriedge-slm/blob/main/agriedge_slm_q4_k_m.gguf")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed: https://huggingface.co/vedantjadhav701/agriedge-slm/blob/main/agriedge_slm_q4_k_m.gguf
